# radial growth vs historical evolution (budapest)

In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    # Font
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,

    # Lines and markers
    "lines.linewidth": 1.2,
    "lines.markersize": 4,

    # Axes
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,

    # Layout
    "figure.constrained_layout.use": True,
})


In [ ]:
import utca
import osmnx as ox
import shapely

import matplotlib.pyplot as plt
import matplotlib as mpl

In [ ]:
G = ox.load_graphml('output/bp_simplified.graphml')
G = utca.prepare_graph(G)

In [ ]:
edges = ox.graph_to_gdfs(G, nodes=False)

In [ ]:
joined = utca.join_historical_streets(edges, fill_na=True)

Not the correct, final map!

In [ ]:
gdf = joined
#gdf["date_num"] = gdf["date"].astype("int64")
gdf["date_num"] = gdf["date"].dt.strftime("%Y")

cm = 1 / 2.54
#fig, ax = plt.subplots(figsize=(8, 6))
cmap = 'plasma'
fig, ax = plt.subplots(figsize=(9*cm, 6*cm))

gdf.plot(
    ax=ax,
    column="date_num",
    cmap=cmap,
    linewidth=0.3,
    #alpha=0.8,
    legend=False,
    missing_kwds={'color': 'k'},
)
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

# create colorbar manually
norm = mpl.colors.Normalize(
    vmin=gdf["date_num"].min(),
    vmax=gdf["date_num"].max()
)
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm._A = []

cbar = fig.colorbar(sm, ax=ax, shrink=0.8)

# format ticks back to dates
#ticks = cbar.get_ticks()
#tick_labels = pd.to_datetime(ticks).strftime("%Y")
#cbar.set_ticks(ticks)
#cbar.set_ticklabels(tick_labels)
cbar.set_label("Date")

plt.show()

In [ ]:
#fig.savefig("output/figs_jan2/bp_map.png", dpi=300)

# get timelines

## historical

In [ ]:
hist = utca.get_timeline(joined)

In [ ]:
hist[['year', 'v', 'n']].plot.line(x='year', subplots=True)

## radial

In [ ]:
lat, lon = 47.4978789, 19.0402383
center = shapely.Point(lon, lat)

In [ ]:
radial = utca.get_radial_timeline(edges, center, 50, 500)

In [ ]:
radial[['radius_meters', 'v', 'n']].plot.line(x='radius_meters', subplots=True)

## plots

### hist

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14*cm, 6*cm))

# n
axs[0].plot('year', 'n', '', data=hist)
axs[0].set_xlabel("year")
axs[0].set_ylabel("$\\overline{n}^*$")#, rotation=0)
axs[0].minorticks_on()
#axs[0].tick_params(axis='x', which='minor', bottom=True)
axs[0].grid()
# v
axs[1].plot('year', 'v', '', data=hist, label='_')
axs[1].set_xlabel("year")
axs[1].set_ylabel("$\\overline{v}^*$")#, rotation=0)
axs[1].minorticks_on()
axs[1].grid()

#fig.savefig("output/figs_maj10/bp_nv.pdf")

In [ ]:
fig, ax = plt.subplots(figsize=(8*cm, 7*cm))
ax.plot(hist['n'], hist['v'], label='Historical')
ax.plot(radial['n'], radial['v'], label='Radial')
row = hist.iloc[-1]
ax.scatter(row['n'], row['v'], s=40, color='0.2', zorder=5, label='Current state')
ax.legend()
#ax.spines['right'].set_visible(False)
#ax.spines['top'].set_visible(False)
ax.set_xlabel("$\\overline{n}^*$")
ax.set_ylabel("$\\overline{v}^*$")#, rotation=0)
#ax.set_title("Evolution of the Street Network of Budapest")
#fig.savefig("output/figs_maj10/bp_rad_hist.pdf")
plt.show()

## radial

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14*cm, 6*cm))

# n
axs[0].plot('radius_meters', 'n', '', data=radial)
axs[0].set_xlabel("Radius (m)")
axs[0].set_ylabel("$\\overline{n}^*$")#, rotation=0)
axs[0].minorticks_on()
#axs[0].tick_params(axis='x', which='minor', bottom=True)
axs[0].grid()
# v
axs[1].plot('radius_meters', 'v', '', data=radial, label='_')
axs[1].set_xlabel("Radius (m)")
axs[1].set_ylabel("$\\overline{v}^*$")#, rotation=0)
axs[1].minorticks_on()
axs[1].grid()

#fig.savefig("output/figs_maj10/bp_rad_nv.pdf")
plt.show()